# An improved PINN framework integrating localized collocation scheme and PIKF

**Paper:** Xi, Q., Xu, W., Cvetkovic, M., Poljak, D., Rabczuk, T., Fu, Z. (2026). *An improved PINN framework integrating localized collocation scheme and PIKF.* arXiv:2606.02585.

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/An improved PINN framework integrating localized.pdf`

## Como se usan las PINNs en este paper

El paper propone **LPIKFNN**, una variante de PINN que **elimina la diferenciacion automatica** para construir el residuo de la EDP. En vez de derivar la salida de la red respecto a sus entradas, LPIKFNN usa **funciones nucleo fisicamente informadas (PIKF)** &mdash; funciones que ya satisfacen la EDP homogenea por construccion (p.ej. soluciones fundamentales, funciones de Green, o en este caso funciones armonicas de Helmholtz) &mdash; para construir, para cada nodo de entrenamiento, un **esquema de colocacion localizado** (Fig. 2): el valor del campo en un nodo se expresa como combinacion lineal de PIKFs centradas en sus nodos vecinos (Eq. 5):

$$u(\mathbf{x}^{(l)})=\sum_{j=1}^M \alpha_j F(\mathbf{x}^{(l)},\mathbf{s}_j)=\mathbf{f}\boldsymbol\alpha_f,\qquad \boldsymbol\alpha_f=\mathbf{F}^{-1}\mathbf{u}$$

Sustituyendo, se obtiene una formula de interpolacion local **fija** (Eq. 8): $u(\mathbf{x}^{(l)})=\sum_j \omega_j^{(f)} u(\mathbf{x}_j)$, donde los pesos $\omega_j^{(f)}$ **dependen solo de la geometria de los nodos y del PIKF elegido, no de los parametros de la red**, por lo que se calculan **una sola vez** antes de entrenar. La perdida de la EDP (Eq. 10) compara, para cada nodo, la prediccion de la red en ese punto contra la reconstruccion ponderada de sus vecinos:

$$\Xi_{pde}=\sum_l \Big|u(\hat{\mathbf{x}}^{(l)})-\sum_j\omega_j^{(f)}u(\hat{\mathbf{x}}_j)\Big|^2$$

Como los pesos $\omega_j$ estan precalculados, esta perdida es una simple funcion de los **valores de salida** de la red en distintos puntos (una diferencia ponderada, sin necesidad de retropropagar derivadas de la red respecto a sus entradas), lo que reduce mucho el costo de entrenamiento frente a una PINN estandar y evita la inestabilidad numerica de la diferenciacion automatica en problemas de alto numero de onda.

Este cuaderno reproduce fielmente este mecanismo para un **problema de Helmholtz 1D de alto numero de onda** (uno de los benchmarks explicitos del paper): $u''(x)+k^2u(x)=0$, usando como PIKF la propia solucion armonica $F(x,s)=\cos(k(x-s))$ (que satisface $F''+k^2F=0$ por construccion), con esquema de colocacion localizado (4 vecinos mas cercanos por nodo) y pesos $\omega_j$ precalculados una sola vez.

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo propio, ni se encontro uno especifico al buscar en GitHub. Como referencia general del framework PINN base sobre el que se construye esta variante:

- **maziarraissi/PINNs** &mdash; https://github.com/maziarraissi/PINNs

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Problema de Helmholtz 1D de alto numero de onda: $u''+k^2u=0$, $u(x)=\sin(kx)$

In [ ]:
k_wave = 20.0  # numero de onda alto, benchmark tipico del paper
N = 61
x_nodes = np.linspace(0, 1, N)

def exact_u(x):
    return np.sin(k_wave * x)

plt.figure(figsize=(7, 3))
plt.plot(x_nodes, exact_u(x_nodes))
plt.xlabel('x'); plt.ylabel('u(x)'); plt.title(f'Solucion exacta (k={k_wave:.0f}, alta oscilacion)')
plt.show()

## 2. PIKF (funcion armonica de Helmholtz) y precalculo de los pesos $\omega_j$ del esquema de colocacion localizado (Eq. 5-9)

$F(x,s)=\cos(k(x-s))$ satisface exactamente $F''+k^2F=0$, por lo que cualquier combinacion lineal de PIKFs ya es (localmente) una solucion de la EDP homogenea. Los pesos se calculan **una sola vez**, antes de entrenar, y no dependen de la red.

In [ ]:
def pikf(x, s):
    return np.cos(k_wave * (x - s))

M_local = 4  # tamano del dominio local (numero de vecinos usados como PIKFs fuente)

def local_neighbors(i, N, M):
    half = M // 2
    lo = max(0, i - half)
    hi = min(N, lo + M)
    lo = max(0, hi - M)
    idx = [j for j in range(lo, hi) if j != i]
    return idx[:M]

# Precalculo de pesos omega_j (Eq. 8) para cada nodo interior.
# Se usa la pseudo-inversa (via SVD) en vez de la inversa exacta: la base coseno pura es
# simetrica, y para vecindarios simetricos respecto al nodo central la matriz F_local puede
# quedar mal condicionada o singular (una combinacion antisimetrica de coeficientes produce
# respuesta nula en los 4 puntos simetricos); la pseudo-inversa da la solucion de norma
# minima de forma robusta en ese caso, sin alterar el mecanismo del esquema (Eq. 7-8).
pde_indices = list(range(1, N - 1))
omega_list = []
neighbor_list = []
for i in pde_indices:
    nb = local_neighbors(i, N, M_local)
    F_local = np.array([[pikf(x_nodes[a], x_nodes[b]) for b in nb] for a in nb])
    f_row = np.array([pikf(x_nodes[i], x_nodes[b]) for b in nb])
    omega = f_row @ np.linalg.pinv(F_local)      # Eq. (8): u(x_i) = sum_j omega_j u(x_j)
    omega_list.append(torch.tensor(omega, dtype=torch.float32, device=device))
    neighbor_list.append(nb)

print(f'Pesos precalculados para {len(pde_indices)} nodos interiores, '
      f'cada uno con {M_local} vecinos locales.')

## 3. Red LPIKFNN y perdida (Eq. 10-13): residuo de la EDP **sin diferenciacion automatica**

In [ ]:
class LPIKFNN(nn.Module):
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


model = LPIKFNN().to(device)
x_all = torch.tensor(x_nodes, dtype=torch.float32, device=device).view(-1, 1)


def compute_loss(model):
    u_all = model(x_all).squeeze(-1)   # una unica pasada hacia adelante para todos los nodos

    # Xi_pde (Eq. 10): sin gradientes, solo combinaciones lineales fijas de u_all
    pde_res = []
    for idx, i in enumerate(pde_indices):
        u_recon = torch.dot(omega_list[idx], u_all[neighbor_list[idx]])
        pde_res.append(u_all[i] - u_recon)
    pde_res = torch.stack(pde_res)
    xi_pde = torch.sum(pde_res**2)

    # Xi_bc: condiciones de Dirichlet en los extremos
    u0, u1 = u_all[0], u_all[-1]
    xi_bc = (u0 - exact_u(x_nodes[0]))**2 + (u1 - exact_u(x_nodes[-1]))**2

    N_pde, N_bc = len(pde_indices), 2
    return (xi_pde + xi_bc) / (N_pde + N_bc)

## 4. Entrenamiento (Adam; el paper usa L-BFGS)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
history = []
for epoch in range(4000):
    optimizer.zero_grad()
    loss = compute_loss(model)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e}')

## 5. Resultados: comparacion con la solucion exacta de alto numero de onda

In [ ]:
with torch.no_grad():
    u_pred = model(x_all).cpu().numpy().flatten()
u_ex = exact_u(x_nodes)
err = 100 * np.linalg.norm(u_pred - u_ex) / np.linalg.norm(u_ex)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x_nodes, u_ex, label='Exacta', linewidth=2)
axes[0].plot(x_nodes, u_pred, '--', label='LPIKFNN')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)')
axes[0].set_title(f'LPIKFNN vs exacta (error relativo L2 = {err:.2f}%)')
axes[0].legend()

axes[1].semilogy(history)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss (escala log)')
axes[1].set_title('Convergencia')
plt.tight_layout()
plt.show()